<a href="https://colab.research.google.com/github/antara2002nigudkar/ML-safety-file/blob/main/clean_and_fgsm_recall_pedestrain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!ls "/content/drive/MyDrive/Colab Notebooks"

'Copy of train_colab.ipynb'    ML_Safety_Ex_6.ipynb	  test-town-01.zip
'Exercise 3 ML_Safety.ipynb'  ' ML_Safety_Ex_9,8.ipynb'   test.zip
'Exercise 4 ML_Safety.ipynb'   ML_Safety.ipynb		  train.zip
'Exercise 5 ML_Safety.ipynb'   test-fog.zip		  Untitled0.ipynb
'Exercise 7.ipynb'	       test-night.zip		  validation.zip


In [3]:
import os
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import recall_score

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cuda


In [5]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [6]:
import os

zip_path = "/content/drive/MyDrive/Colab Notebooks/test.zip"

print("Exists:", os.path.exists(zip_path))
print("Size:", os.path.getsize(zip_path) if os.path.exists(zip_path) else "Not found")

Exists: True
Size: 312065800


In [7]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/Colab Notebooks/test.zip"
extract_path = "/content"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction complete")

Extraction complete


In [8]:
import os

for root, dirs, files in os.walk("/content"):
    if "rgb-front" in dirs:
        print("Found:", os.path.join(root, "rgb-front"))

Found: /content/test/rgb-front
Found: /content/drive/MyDrive/test-night/rgb-front
Found: /content/drive/MyDrive/test-town-01/rgb-front
Found: /content/drive/MyDrive/test-fog/rgb-front
Found: /content/drive/MyDrive/test/test/rgb-front
Found: /content/drive/MyDrive/validation/rgb-front
Found: /content/drive/.Encrypted/MyDrive/test-night/rgb-front
Found: /content/drive/.Encrypted/MyDrive/test-town-01/rgb-front
Found: /content/drive/.Encrypted/MyDrive/test-fog/rgb-front
Found: /content/drive/.Encrypted/MyDrive/test/test/rgb-front
Found: /content/drive/.Encrypted/MyDrive/validation/rgb-front


In [9]:
test_path = "/content/test/rgb-front"

all_files = sorted(os.listdir(test_path))

print("Total test images:", len(all_files))

Total test images: 3600


In [10]:
import os
import torch
from torch.utils.data import Dataset
from PIL import Image
import pandas as pd
import torchvision.transforms as transforms

class CARLADataset(Dataset):

    def __init__(self, root_dir):

        self.root_dir = root_dir

        self.labels = pd.read_csv(
            os.path.join(root_dir, "labels.csv")
        )

        self.rgb_dir = os.path.join(
            root_dir,
            "rgb-front"
        )

        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        row = self.labels.iloc[idx]

        img_name = str(int(row["frame"])).zfill(6)

        img_path = os.path.join(
            self.rgb_dir,
            f"{img_name}.jpg"
        )

        image = Image.open(img_path).convert("RGB")

        image = self.transform(image)

        label = torch.tensor([
            row["has_traffic_light"],
            row["has_pedestrian"],
            row["has_vehicle"]
        ], dtype=torch.float32)

        return image, label

In [11]:
test_path = "/content/test"

test_dataset = CARLADataset(test_path)

print("Test dataset size:", len(test_dataset))

Test dataset size: 3600


In [12]:
image, label = test_dataset[0]

print("Image shape:", image.shape)
print("Label:", label)

Image shape: torch.Size([3, 224, 224])
Label: tensor([0., 0., 0.])


In [13]:
import torch

torch.manual_seed(42)

indices = torch.randperm(len(test_dataset))[:100]

print("Number of selected images:", len(indices))

Number of selected images: 100


In [14]:
import torch
from torchvision import models

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

def load_model(path):
    model = models.resnet18(pretrained=False)
    model.fc = torch.nn.Linear(512, 1)

    model.load_state_dict(
        torch.load(path, map_location=device)
    )

    model = model.to(device)
    model.eval()

    return model

models_dict = {
    "pedestrian": load_model(
        "/content/drive/MyDrive/ped_model.pth"
    ),

    "vehicle": load_model(
        "/content/drive/MyDrive/vehicle_model.pth"
    ),

    "traffic": load_model(
        "/content/drive/MyDrive/traffic_model.pth"
    )
}

print("All three models loaded successfully.")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


All three models loaded successfully.


In [16]:
import random
from torch.utils.data import Subset, DataLoader

random.seed(42)

indices = random.sample(range(len(test_dataset)), 100)

test_subset = Subset(test_dataset, indices)

test_loader = DataLoader(
    test_subset,
    batch_size=16,
    shuffle=False
)

print("Evaluation samples:", len(test_subset))

Evaluation samples: 100


In [17]:
image, label = test_subset[0]

print("Image shape:", image.shape)
print("Label:", label)

Image shape: torch.Size([3, 224, 224])
Label: tensor([1., 1., 1.])


In [30]:
label_indices = {
    "traffic": 0,
    "pedestrian": 1,
    "vehicle": 2
}

print(label_indices)

{'traffic': 0, 'pedestrian': 1, 'vehicle': 2}


In [31]:
from sklearn.metrics import recall_score

def calculate_clean_recall(model, loader, label_idx):

    model.eval()

    all_predictions = []
    all_labels = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)

            outputs = model(images)

            probabilities = torch.sigmoid(outputs)

            predictions = (
                probabilities >= 0.5
            ).float().squeeze(1)

            true_labels = labels[:, label_idx]

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                true_labels.numpy()
            )

    recall = recall_score(
        all_labels,
        all_predictions,
        zero_division=0
    )

    return recall

In [32]:
from sklearn.metrics import recall_score

def calculate_pedestrian_clean_recall(model, loader):

    model.eval()

    all_predictions = []
    all_labels = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)

            outputs = model(images)

            probabilities = torch.sigmoid(outputs)

            predictions = (
                probabilities >= 0.5
            ).float().squeeze()

            # Pedestrian is column 1
            true_labels = labels[:, 1]

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                true_labels.cpu().numpy()
            )

    recall = recall_score(
        all_labels,
        all_predictions,
        zero_division=0
    )

    return recall

In [34]:
ped_recall = calculate_pedestrian_clean_recall(
    models_dict["pedestrian"],
    test_loader
)

print(f"Pedestrian clean recall: {ped_recall:.4f}")

Pedestrian clean recall: 0.3478


In [35]:
import torch
import torch.nn.functional as F
from sklearn.metrics import recall_score

def calculate_pedestrian_fgsm_recall(model, loader, epsilon=0.05):

    model.eval()

    all_predictions = []
    all_labels = []

    for images, labels in loader:

        images = images.to(device)
        images.requires_grad_(True)

        # Pedestrian labels = column 1
        true_labels = labels[:, 1].float().to(device).unsqueeze(1)

        # Forward pass
        outputs = model(images)

        # FGSM loss
        loss = F.binary_cross_entropy_with_logits(
            outputs,
            true_labels
        )

        # Calculate gradient
        model.zero_grad()
        loss.backward()

        # FGSM attack
        adversarial_images = (
            images + epsilon * images.grad.sign()
        )

        adversarial_images = torch.clamp(
            adversarial_images,
            0,
            1
        ).detach()

        # Prediction on adversarial images
        with torch.no_grad():

            adv_outputs = model(adversarial_images)

            probabilities = torch.sigmoid(
                adv_outputs
            )

            predictions = (
                probabilities >= 0.5
            ).float().squeeze(1)

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            true_labels.squeeze(1).cpu().numpy()
        )

    recall = recall_score(
        all_labels,
        all_predictions,
        zero_division=0
    )

    return recall

In [36]:
ped_fgsm_recall = calculate_pedestrian_fgsm_recall(
    models_dict["pedestrian"],
    test_loader,
    epsilon=0.05
)

print(
    f"Pedestrian FGSM recall (ε=0.05): "
    f"{ped_fgsm_recall:.4f}"
)

Pedestrian FGSM recall (ε=0.05): 0.1304
